In [0]:

# ==========================================
#  LOAD AND FILTER DATA
# ==========================================

from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    sum as spark_sum,
    avg,
    when,
    round
)

# Load Silver tables
matches_df = spark.table(
    "workspace.default.silver_matches"
)

teams_df = spark.table(
    "workspace.default.silver_teams"
)

competition_df = spark.table(
    "workspace.default.silver_competition"
)

# Keep only completed matches with valid scores
finished_df = matches_df.filter(
    (col("match_status") == "FINISHED") &
    col("home_goals").isNotNull() &
    col("away_goals").isNotNull()
)

# Validate unique match IDs
assert finished_df.count() == finished_df.select(
    "match_id"
).distinct().count(), "Duplicate matches detected!"

print("Total fixtures:", matches_df.count())
print("Completed matches:", finished_df.count())
print("Teams:", teams_df.count())
print("Competition records:", competition_df.count())


In [0]:
# ==========================================
# CALCULATE LEAGUE STATISTICS
# ==========================================

match_summary_df = finished_df.groupBy(
    "season_id"
).agg(

    # Completed matches
    count("*").alias("completed_matches"),

    # Total goals scored
    spark_sum(
        col("home_goals") + col("away_goals")
    ).alias("total_goals"),

    # Average goals per match
    round(
        avg(col("home_goals") + col("away_goals")),
        2
    ).alias("avg_goals_per_match"),

    # Home wins
    spark_sum(
        when(
            col("home_goals") > col("away_goals"), 1
        ).otherwise(0)
    ).alias("home_wins"),

    # Away wins
    spark_sum(
        when(
            col("away_goals") > col("home_goals"), 1
        ).otherwise(0)
    ).alias("away_wins"),

    # Draws
    spark_sum(
        when(
            col("home_goals") == col("away_goals"), 1
        ).otherwise(0)
    ).alias("draws"),

    # Home team clean sheets
    spark_sum(
        when(
            col("away_goals") == 0, 1
        ).otherwise(0)
    ).alias("home_clean_sheets"),

    # Away team clean sheets
    spark_sum(
        when(
            col("home_goals") == 0, 1
        ).otherwise(0)
    ).alias("away_clean_sheets")
)

display(match_summary_df)


In [0]:

# ==========================================
# BUILD GOLD LEAGUE SUMMARY
# ==========================================

# Count all scheduled fixtures per season
fixture_summary_df = matches_df.groupBy(
    "season_id"
).agg(
    count("*").alias("total_fixtures")
)

# Count unique Premier League teams
team_count_df = teams_df.agg(
    countDistinct("team_id").alias("total_teams")
)

# Join the datasets
gold_df = (
    competition_df
    .join(
        fixture_summary_df,
        "season_id",
        "left"
    )
    .join(
        match_summary_df,
        "season_id",
        "left"
    )
    .crossJoin(team_count_df)

    # Calculate remaining fixtures
    .withColumn(
        "remaining_fixtures",
        col("total_fixtures") -
        col("completed_matches")
    )

    # Calculate total clean sheets
    .withColumn(
        "total_clean_sheets",
        col("home_clean_sheets") +
        col("away_clean_sheets")
    )

    # Select final Gold columns
    .select(
        "season_id",
        "competition_name",
        "season_start",
        "season_end",
        "current_matchday",
        "total_teams",
        "total_fixtures",
        "completed_matches",
        "remaining_fixtures",
        "total_goals",
        "avg_goals_per_match",
        "home_wins",
        "away_wins",
        "draws",
        "home_clean_sheets",
        "away_clean_sheets",
        "total_clean_sheets"
    )
)

print("Gold league summary DataFrame created!")
display(gold_df)


In [0]:

# ==========================================
# VALIDATE AND SAVE GOLD TABLE
# ==========================================

# Check that we have one current season
assert gold_df.count() == 1, \
    "Expected one current Premier League season!"

row = gold_df.first()

# Validate fixture counts
assert (
    row["completed_matches"] +
    row["remaining_fixtures"]
    == row["total_fixtures"]
), "Fixture counts do not reconcile!"

# Validate match results
assert (
    row["home_wins"] +
    row["away_wins"] +
    row["draws"]
    == row["completed_matches"]
), "Match results do not reconcile!"

# Validate clean sheets
assert (
    row["total_clean_sheets"]
    == row["home_clean_sheets"] +
    row["away_clean_sheets"]
), "Clean sheet counts do not reconcile!"

# Validate team count
assert row["total_teams"] == 20, \
    "Unexpected Premier League team count!"

print("Gold league summary validation passed!")

# ==========================================
# SAVE AS DELTA TABLE
# ==========================================

(
    gold_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.default.gold_league_summary"
    )
)

print("Gold league summary table saved successfully!")


In [0]:
%sql

SELECT
    competition_name,
    season_id,
    total_teams,
    total_fixtures,
    completed_matches,
    remaining_fixtures,
    total_goals,
    avg_goals_per_match,
    home_wins,
    away_wins,
    draws,
    home_clean_sheets,
    away_clean_sheets,
    total_clean_sheets
FROM workspace.default.gold_league_summary;
